# Fused Spectral Index Computation with Custom Pallas Kernels on TorchTPU

This tutorial walks through accelerating **NDVI** and **NDWI** computation on full Sentinel-2 10m tiles (`10980 x 10980` pixels) on Cloud TPUs using a custom 2D-tiled **Pallas** kernel via `torch.tpu.pallas`.

### Why Fuse Spectral Indices?
Element-wise remote sensing formulas are heavily **memory-bandwidth bound**:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red} + \epsilon}, \qquad \text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR} + \epsilon}$$

In eager PyTorch execution, each arithmetic operation reads from and writes back to High Bandwidth Memory (HBM). By fusing both indices into a single 2D-tiled Pallas kernel:
1. Each `(block_h, block_w)` tile of Red (B04), Green (B03), and NIR (B08) is loaded **once** from HBM into TPU VMEM (SRAM).
2. Inputs are upcast to `float32` inside the Vector Processing Unit (VPU) for numerical stability when using `bfloat16`.
3. Both NDVI and NDWI are computed simultaneously and written back to HBM in a single pass (theoretical minimum: **3 reads + 2 writes**).

## 1. Environment Setup & Imports

Make sure you are running this notebook on a **Cloud TPU VM** with TorchTPU and TPU-enabled JAX installed.

In [ ]:
import sys
from pathlib import Path

# Ensure the repository root is on sys.path when running from notebooks/
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import jax
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch_tpu  # Registers the native 'tpu' backend

from src.baseline import compute_indices_eager, get_compiled_indices_fn
from src.data_utils import (
    PB0400_RADIO_ADD_OFFSET,
    SENTINEL2_10M_SHAPE,
    generate_synthetic_bands,
    load_sentinel2_bands,
    save_index_geotiff,
)
from src.pallas_kernel import compute_indices_pallas
from benchmarks.bench_throughput import run_benchmarks

DEVICE = torch.device("tpu")

print(f"Repo root : {REPO_ROOT}")
print(f"PyTorch   : {torch.__version__}")
print(f"JAX       : {jax.__version__} ({jax.devices()})")
print(f"Target dev: {DEVICE}")

## 2. Synthetic Reflectance Bands (or Real Sentinel-2 L2A Data)

We start with a `1024 x 1024` spatially coherent synthetic tile (`realistic=True`) so we can inspect the resulting NDVI/NDWI maps visually. The commented block shows how to load real Sentinel-2 L2A GeoTIFF/JP2 bands using `load_sentinel2_bands()`.

In [ ]:
DEMO_SHAPE = (1024, 1024)
DEMO_DTYPE = torch.bfloat16

red, green, nir = generate_synthetic_bands(
    shape=DEMO_SHAPE,
    dtype=DEMO_DTYPE,
    device=DEVICE,
    seed=42,
    realistic=True,
)

# --- Optional: Load real Sentinel-2 L2A bands (10m B04, B03, B08) ---
# red, green, nir, profile = load_sentinel2_bands(
#     red_path="data/T32TQM_B04_10m.jp2",
#     green_path="data/T32TQM_B03_10m.jp2",
#     nir_path="data/T32TQM_B08_10m.jp2",
#     dtype=torch.bfloat16,
#     device=DEVICE,
#     scale_factor=10000.0,
#     offset=PB0400_RADIO_ADD_OFFSET,  # Use -1000.0 for Processing Baseline 04.00+
#     window=(0, 0, 2048, 2048),       # Optional crop: (col_off, row_off, width, height)
# )

print(f"Band shape: {tuple(red.shape)}, dtype: {red.dtype}, device: {red.device}")

## 3. Running Eager, `torch.compile`, and Custom Pallas

Let's compute `(ndvi, ndwi)` using all three execution modes:
1. **Eager (`compute_indices_eager`)** — standard PyTorch element-wise ops.
2. **Compiled (`get_compiled_indices_fn`)** — `torch.compile(compute_indices_eager, backend="tpu")`.
3. **Pallas (`compute_indices_pallas`)** — our custom 2D-tiled VMEM kernel with `block_shape=(128, 128)` (aligned to TPU `8x128` vector sublanes).

In [ ]:
compiled_fn = get_compiled_indices_fn()

eager_ndvi, eager_ndwi = compute_indices_eager(red, green, nir)
comp_ndvi, comp_ndwi = compiled_fn(red, green, nir)
pal_ndvi, pal_ndwi = compute_indices_pallas(red, green, nir, block_shape=(128, 128))

print("NDVI range (Pallas):", float(pal_ndvi.min()), "to", float(pal_ndvi.max()))
print("NDWI range (Pallas):", float(pal_ndwi.min()), "to", float(pal_ndwi.max()))

## 4. Numerical Equivalence, Auto-Padding & Zero-Division Stability

Full Sentinel-2 tiles (`10980 x 10980`) are **not** exact multiples of `128`, `256`, or `512`. `compute_indices_pallas` automatically zero-pads inputs to the next block multiple and slices outputs back to `(H, W)`.

Below we verify:
- Exact match within dtype tolerances (`float32` and `bfloat16`) on a non-multiple shape (`1098 x 1098`).
- No `NaN` or `Inf` on all-zero nodata pixels.

In [ ]:
for dtype, rtol, atol in [(torch.float32, 1e-5, 1e-6), (torch.bfloat16, 1e-2, 1e-2)]:
    r_t, g_t, n_t = generate_synthetic_bands((1098, 1098), dtype=dtype, device=DEVICE)
    ref_ndvi, ref_ndwi = compute_indices_eager(r_t, g_t, n_t)
    p_ndvi, p_ndwi = compute_indices_pallas(r_t, g_t, n_t, block_shape=(128, 128))

    assert p_ndvi.shape == (1098, 1098) and p_ndvi.dtype == dtype
    torch.testing.assert_close(p_ndvi.cpu(), ref_ndvi.cpu(), rtol=rtol, atol=atol)
    torch.testing.assert_close(p_ndwi.cpu(), ref_ndwi.cpu(), rtol=rtol, atol=atol)
    print(f"[PASS] {dtype} on non-multiple shape (1098, 1098)")

# All-zero nodata edge case
zeros = torch.zeros((300, 300), dtype=torch.bfloat16, device=DEVICE)
z_ndvi, z_ndwi = compute_indices_pallas(zeros, zeros, zeros, block_shape=(128, 128))
assert torch.isfinite(z_ndvi.cpu()).all() and torch.isfinite(z_ndwi.cpu()).all()
assert torch.all(z_ndvi.cpu() == 0.0) and torch.all(z_ndwi.cpu() == 0.0)
print("[PASS] Zero-reflectance stability (no NaN/Inf)")

## 5. Visualizing the Computed NDVI & NDWI Maps

Let's plot a false-color composite (NIR, Red, Green) alongside the fused Pallas NDVI and NDWI outputs.

In [ ]:
nir_np = nir.cpu().float().numpy()
red_np = red.cpu().float().numpy()
green_np = green.cpu().float().numpy()
false_color = torch.stack([nir.cpu().float(), red.cpu().float(), green.cpu().float()], dim=-1).clamp(0, 1).numpy()

ndvi_np = pal_ndvi.cpu().float().numpy()
ndwi_np = pal_ndwi.cpu().float().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(false_color)
axes[0].set_title("False Color (NIR / Red / Green)")
axes[0].axis("off")

im1 = axes[1].imshow(ndvi_np, cmap="RdYlGn", vmin=-1.0, vmax=1.0)
axes[1].set_title("NDVI (Pallas Fused)")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(ndwi_np, cmap="BrBG", vmin=-1.0, vmax=1.0)
axes[2].set_title("NDWI (Pallas Fused)")
axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# --- Optional: Export to compressed tiled GeoTIFF when using real Sentinel-2 data ---
# save_index_geotiff(pal_ndvi, "results/ndvi_pallas.tif", profile)
# save_index_geotiff(pal_ndwi, "results/ndwi_pallas.tif", profile)

## 6. Throughput & Effective HBM Bandwidth Benchmark

Now let's benchmark a full **Sentinel-2 10m tile (`10980 x 10980`)** across `float32` and `bfloat16`, sweeping Pallas tile shapes `(128, 128)`, `(256, 256)`, and `(512, 512)`.

Effective HBM bandwidth is measured against the fused theoretical minimum (3 reads + 2 writes):

$$\text{Effective BW (GB/s)} = \frac{5 \cdot H \cdot W \cdot \text{sizeof}(\text{dtype}) / 10^9}{\text{median\_ms} / 10^3}$$

In [ ]:
rows = run_benchmarks(
    shapes=[SENTINEL2_10M_SHAPE],
    dtypes=[torch.float32, torch.bfloat16],
    block_shapes=[(128, 128), (256, 256), (512, 512)],
    warmup=5,
    iters=20,
)

df = pd.DataFrame(rows)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, metric, ylabel, title in [
    (axes[0], "median_ms", "Median Latency (ms)", "Latency on 10980x10980 Tile (Lower is Better)"),
    (axes[1], "bandwidth_gb_s", "Effective HBM Bandwidth (GB/s)", "Effective HBM Bandwidth (Higher is Better)"),
]:
    pivot = df.pivot(index="method", columns="dtype", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=20)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Profiling with XProf (TensorBoard)

To inspect kernel fusion directly on the TPU timeline, capture XProf traces from the command line:

```bash
python -m benchmarks.profile_xprof \
  --shape 10980x10980 \
  --dtype bfloat16 \
  --block-shape 256x256 \
  --logdir results/xprof_traces

tensorboard --logdir results/xprof_traces --port 6006 --bind_all
```

In the **XProf Trace Viewer**, compare the `eager` run (multiple separate element-wise HLO/TPU kernels with intermediate HBM transfers) against the `pallas` run (a single fused custom-call kernel executing directly on VMEM tiles).